In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [4]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          "Let's "
                                                                          'reschedule '
                                                                          'for '
                                                                          'later '
                                                                          'in '
                                                                          'the '
              

In [8]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John, no problem at all. Let's reschedule for later in the week. Best, Seán."}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John, no problem at all. Let\'s reschedule for later in the week. Best, Seán."}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='12153798b11944b5e13bf67c3fc3cc3f')]


In [9]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem at all. Let's reschedule for later in the week. Best, Seán.


## Approve

In [10]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response['messages'])

[HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='f48b55ef-ca77-4548-b8e6-be8625e8414e'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_11966327100644887308': 'EmAKXgFpFH0Tns5Vaa77w3MyJQfbByVRv/u60Aopa+5E6EPj2o9YxAhiG30wDvaHndifdNob7C0jUCbAESFhwRweR+aNFH52xuFmWdcYZDmUm0aFzqrY4xUFSG9RVItWRVs='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cefb-410d-7a22-81e8-208219ab6955-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_11966327100644887308', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 92, 'output_tokens': 10, 'total_tokens': 102, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content="Hi Seán, I'm

## Reject

In [17]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='f48b55ef-ca77-4548-b8e6-be8625e8414e'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_11966327100644887308': 'EmAKXgFpFH0Tns5Vaa77w3MyJQfbByVRv/u60Aopa+5E6EPj2o9YxAhiG30wDvaHndifdNob7C0jUCbAESFhwRweR+aNFH52xuFmWdcYZDmUm0aFzqrY4xUFSG9RVItWRVs='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cefb-410d-7a22-81e8-208219ab6955-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_11966327100644887308', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'

In [22]:
print(response['messages'])

[HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='f48b55ef-ca77-4548-b8e6-be8625e8414e'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_11966327100644887308': 'EmAKXgFpFH0Tns5Vaa77w3MyJQfbByVRv/u60Aopa+5E6EPj2o9YxAhiG30wDvaHndifdNob7C0jUCbAESFhwRweR+aNFH52xuFmWdcYZDmUm0aFzqrY4xUFSG9RVItWRVs='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cefb-410d-7a22-81e8-208219ab6955-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_11966327100644887308', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 92, 'output_tokens': 10, 'total_tokens': 102, 'input_token_details': {'cache_read': 0}}), ToolMessage(content="Hi Seán, I'm g

## Edit

In [20]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response['messages'])

[HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='f48b55ef-ca77-4548-b8e6-be8625e8414e'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_11966327100644887308': 'EmAKXgFpFH0Tns5Vaa77w3MyJQfbByVRv/u60Aopa+5E6EPj2o9YxAhiG30wDvaHndifdNob7C0jUCbAESFhwRweR+aNFH52xuFmWdcYZDmUm0aFzqrY4xUFSG9RVItWRVs='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cefb-410d-7a22-81e8-208219ab6955-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_11966327100644887308', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 92, 'output_tokens': 10, 'total_tokens': 102, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content="Hi Seán, I'm